In [6]:
# ============================================================
# CELL 1 — Imports
# All dependencies loaded once here; no re-imports in later cells
# ============================================================
import os
import json
import shutil
import random
import numpy as np
from collections import Counter

from ase.io import read                          # read QE .pwi input files
from pymatgen.core.structure import Structure    # pymatgen Structure object
from monty.serialization import loadfn, dumpfn  # convenient JSON I/O


In [7]:
# ============================================================
# CELL 2 — Isolated-atom reference energies → eref_90 dict
#
# Goal: build eref_90[label] = sum of isolated-atom energies (eV)
#       for each 90-atom (3×3×2) supercell composition.
#
# Why subtract isolated-atom energies?
#   NEP training uses cohesive/binding energies, not raw DFT total
#   energies.  Subtracting the sum of free-atom energies centres the
#   energy scale around zero and makes learning easier.
#
# Source of atom energies:
#   Single-atom spin-polarised QE calculations, stored in Ry.
#   Converted to eV with 1 Ry = 13.6057 eV (QE convention).
# ============================================================

# Raw isolated-atom total energies from QE (Ry)
energies_ry = {
    "Sb": -140.74358039,
    "Bi": -154.95167999,
    "Te": -174.52034190,
}

RY_TO_EV = 13.6057                                         # QE conversion factor
E_atom   = {k: v * RY_TO_EV for k, v in energies_ry.items()}  # eV

# Print converted values for reference
print("Isolated-atom energies:")
for el in energies_ry:
    print(f"  {el}: {energies_ry[el]:.8f} Ry  =  {E_atom[el]:.6f} eV")
print()

def calc_eref(n_Bi, n_Sb, n_Te):
    """Return sum of isolated-atom energies (eV) for a given composition."""
    return (
        n_Bi * E_atom["Bi"] +
        n_Sb * E_atom["Sb"] +
        n_Te * E_atom["Te"]
    )

# ------------------------------------------------------------------
# Build eref_90: read 90-atom QE input files to get composition,
# then compute the reference energy for each alloy endpoint.
#
# Note: Sb2Te3 has no 90-atom .pwi file, so it is added manually
#       (Bi:0, Sb:36, Te:54 → same 3×3×2 supercell logic).
# ------------------------------------------------------------------
folder  = "/home/ashwani/BiSbTe_AlloyTransport/BiSbTe_alloy_endpoints"
eref_90 = {}  # will be used in every subsequent cell

for fname in sorted(os.listdir(folder)):
    if not (fname.endswith(".pwi") and "primitive" in fname):
        continue
    atoms = read(os.path.join(folder, fname), format="espresso-in")
    if len(atoms) != 90:          # skip 5-atom primitive cells
        continue
    count = Counter(atoms.get_chemical_symbols())
    # derive clean label: e.g. "Bi2Te3", "BiSbTe20", ...
    label = (fname
             .replace("espresso_", "")
             .replace("_332_supercell_primitive.pwi", "")
             .replace("_primitive.pwi", ""))
    eref_90[label] = calc_eref(
        count.get("Bi", 0), count.get("Sb", 0), count.get("Te", 0)
    )

# Sb2Te3 endpoint added manually (no 90-atom .pwi exists)
eref_90["Sb2Te3"] = calc_eref(0, 36, 54)

print("eref_90 — 90-atom reference energies (eV):")
for label, eref in eref_90.items():
    print(f"  {label}: {eref:.4f} eV")


Isolated-atom energies:
  Sb: -140.74358039 Ry  =  -1914.914932 eV
  Bi: -154.95167999 Ry  =  -2108.226072 eV
  Te: -174.52034190 Ry  =  -2374.471416 eV

eref_90 — 90-atom reference energies (eV):
  Bi2Te3: -204117.5951 eV
  BiSbTe20: -202764.4171 eV
  BiSbTe40: -201411.2391 eV
  BiSbTe60: -199864.7500 eV
  BiSbTe80: -198511.5720 eV
  Sb2Te3: -197158.3940 eV


In [12]:
# ============================================================
# CELL 3 — Load all JSON datasets and subtract reference energies
#
# Three dataset types are included:
#   1. random_disp_0.3A  — random atomic displacements (200 structs each)
#   2. shear_strain      — shear-strained supercells (50 structs each)
#   3. uniaxial_strain   — uniaxial strain along X/Y/Z (11 structs each)
#
# For each file label we map to the correct eref_90 key so the right
# composition reference is subtracted from every structure.
#
# Output arrays (used in Cell 4 for XYZ writing):
#   total_structures  : List[Structure]   — pymatgen Structure objects
#   total_energies    : np.ndarray (eV)   — cohesive energies (ref subtracted)
#   total_forces      : List[array]       — forces in eV/Å
#   total_stresses    : List[array]       — virial stress in GPa (Voigt-6)
# ============================================================

# ------------------------------------------------------------------
# Map each files{} label → eref_90 key
# The tag after '/' in the label is matched by prefix against this dict.
# Two naming conventions exist in the files dict:
#   - random_disp uses "BiTeSb_XX"   → maps to "BiSbTeXX"
#   - shear/uniaxial use bare "XX"   → maps to "BiSbTeXX"
# ------------------------------------------------------------------
label_map = {
    "Bi2Te3"   : "Bi2Te3",
    "BiTeSb_20": "BiSbTe20",
    "BiTeSb_40": "BiSbTe40",
    "BiTeSb_60": "BiSbTe60",
    "BiTeSb_80": "BiSbTe80",
    "Sb2Te3"   : "Sb2Te3",
    "20"       : "BiSbTe20",
    "40"       : "BiSbTe40",
    "60"       : "BiSbTe60",
    "80"       : "BiSbTe80",
}

def get_eref_key(file_label):
    """Extract the eref_90 key from a files{} label string.
    e.g. 'shear/20_1pct_0.3ptb' → tag='20_1pct_0.3ptb' → 'BiSbTe20'
    """
    tag = file_label.split("/")[-1]   # strip prefix like 'shear/'
    for k, v in label_map.items():
        if tag.startswith(k):
            return v
    raise ValueError(f"Cannot map '{file_label}' to an eref_90 key")

def get_strain_metadata(file_label):
    """
    Parse strain value, perturbation amplitude, and strain direction
    from the file label string.

    Examples:
      'shear/20_3pct_0.3ptb'      → strain_pct=3.0,  perturbation=0.3, direction=None
      'shear/Bi2Te3_1pct_0ptb'    → strain_pct=1.0,  perturbation=0.0, direction=None
      'uniaxial/60_X'             → strain_pct=None, perturbation=0.0, direction=X
      'random_disp/BiTeSb_40'     → strain_pct=None, perturbation=0.3, direction=None
    """
    tag       = file_label.split("/")[-1]   # e.g. '20_3pct_0.3ptb', 'Bi2Te3_X'
    dtype     = file_label.split("/")[0]    # 'random_disp', 'shear', 'uniaxial'

    strain_pct    = None
    perturbation  = 0.0
    direction     = None

    if dtype == "random_disp":
        # random displacement, no strain, fixed 0.3 Å perturbation
        perturbation = 0.3

    elif dtype == "shear":
        # e.g. "20_3pct_0.3ptb" or "Bi2Te3_1pct_0ptb"
        import re
        m_strain = re.search(r'(\d+)pct', tag)
        m_ptb    = re.search(r'perturb_(\d+\.?\d*)|_(\d+\.?\d*)ptb', tag)
        if m_strain:
            strain_pct = float(m_strain.group(1))
        # parse perturbation from e.g. "0.3ptb" or "0ptb"
        m_ptb2 = re.search(r'_(\d+\.?\d*)ptb', tag)
        if m_ptb2:
            perturbation = float(m_ptb2.group(1))

    elif dtype == "uniaxial":
        # e.g. "60_X" or "Bi2Te3_Y"
        import re
        m_dir = re.search(r'_(X|Y|Z)$', tag)
        if m_dir:
            direction = m_dir.group(1)
        perturbation = 0.0   # uniaxial has no atomic perturbation

    return strain_pct, perturbation, direction

# ------------------------------------------------------------------
# File registry
# Keys  = short human-readable labels (also used for eref mapping)
# Values = full paths to JSON files produced by pymatgen/atomate
# ------------------------------------------------------------------
base_path = "/home/ashwani/BiSbTe_AlloyTransport/training_data_json_format/"

files = {
    # -------- random atomic displacements (0.3 Å amplitude) --------
    "random_disp/Bi2Te3"        : base_path + "random_disp_0.3A/Bi2Te3.json",
    "random_disp/BiTeSb_20"     : base_path + "random_disp_0.3A/BiTeSb_20.json",
    "random_disp/BiTeSb_40"     : base_path + "random_disp_0.3A/BiTeSb_40.json",
    "random_disp/BiTeSb_60"     : base_path + "random_disp_0.3A/BiTeSb_60.json",
    "random_disp/BiTeSb_80"     : base_path + "random_disp_0.3A/BiTeSb_80.json",
    "random_disp/Sb2Te3"        : base_path + "random_disp_0.3A/Sb2Te3.json",
    # -------- shear strain (1% and 3%, with/without atomic perturbation) --------
    "shear/Bi2Te3_1pct_0.3ptb"  : base_path + "shear_strain/Bi2Te3_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/Bi2Te3_1pct_0ptb"    : base_path + "shear_strain/Bi2Te3_1_pct_perturb_0_ptb_lattice.json",
    "shear/Bi2Te3_3pct_0.3ptb"  : base_path + "shear_strain/Bi2Te3_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/Bi2Te3_3pct_0ptb"    : base_path + "shear_strain/Bi2Te3_3_pct_perturb_0_ptb_lattice.json",
    "shear/20_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_20_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/20_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_20_1_pct_perturb_0_ptb_lattice.json",
    "shear/20_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_20_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/20_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_20_3_pct_perturb_0_ptb_lattice.json",
    "shear/40_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_40_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/40_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_40_1_pct_perturb_0_ptb_lattice.json",
    "shear/40_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_40_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/40_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_40_3_pct_perturb_0_ptb_lattice.json",
    "shear/60_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_60_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/60_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_60_1_pct_perturb_0_ptb_lattice.json",
    "shear/60_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_60_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/60_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_60_3_pct_perturb_0_ptb_lattice.json",
    "shear/80_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_80_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/80_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_80_1_pct_perturb_0_ptb_lattice.json",
    "shear/80_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_80_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/80_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_80_3_pct_perturb_0_ptb_lattice.json",
    "shear/Sb2Te3_1pct_0ptb"    : base_path + "shear_strain/Sb2Te3_1_pct_perturb_0_ptb_lattice.json",
    "shear/Sb2Te3_3pct_0ptb"    : base_path + "shear_strain/Sb2Te3_3_pct_perturb_0_ptb_lattice.json",
    "shear/Sb2Te3_1pct_0.3ptb"  : base_path + "shear_strain/Sb2Te3_1_pct_perturb_0.3_ptb_lattice.json",
    # -------- uniaxial strain along X, Y, Z --------
    "uniaxial/Bi2Te3_X"         : base_path + "uniaxial_strain/Bi2Te3_X.json",
    "uniaxial/Bi2Te3_Y"         : base_path + "uniaxial_strain/Bi2Te3_Y.json",
    "uniaxial/Bi2Te3_Z"         : base_path + "uniaxial_strain/Bi2Te3_Z.json",
    "uniaxial/20_X"             : base_path + "uniaxial_strain/BiSbTe_20_pct_X.json",
    "uniaxial/20_Y"             : base_path + "uniaxial_strain/BiSbTe_20_pct_Y.json",
    "uniaxial/20_Z"             : base_path + "uniaxial_strain/BiSbTe_20_pct_Z.json",
    "uniaxial/40_X"             : base_path + "uniaxial_strain/BiSbTe_40_pct_X.json",
    "uniaxial/40_Y"             : base_path + "uniaxial_strain/BiSbTe_40_pct_Y.json",
    "uniaxial/40_Z"             : base_path + "uniaxial_strain/BiSbTe_40_pct_Z.json",
    "uniaxial/60_X"             : base_path + "uniaxial_strain/BiSbTe_60_pct_X.json",
    "uniaxial/60_Y"             : base_path + "uniaxial_strain/BiSbTe_60_pct_Y.json",
    "uniaxial/60_Z"             : base_path + "uniaxial_strain/BiSbTe_60_pct_Z.json",
    "uniaxial/80_X"             : base_path + "uniaxial_strain/BiSbTe_80_pct_X.json",
    "uniaxial/80_Y"             : base_path + "uniaxial_strain/BiSbTe_80_pct_Y.json",
    "uniaxial/80_Z"             : base_path + "uniaxial_strain/BiSbTe_80_pct_Z.json",
    "uniaxial/Sb2Te3_X"         : base_path + "uniaxial_strain/Sb2Te3_X.json",
    "uniaxial/Sb2Te3_Y"         : base_path + "uniaxial_strain/Sb2Te3_Y.json",
    "uniaxial/Sb2Te3_Z"         : base_path + "uniaxial_strain/Sb2Te3_Z.json",
}

# ------------------------------------------------------------------
# Load all JSON files, tag each entry with:
#   _eref_key   → which eref_90 entry to subtract  (e.g. 'BiSbTe40')
#   _file_label → full label for data_type parsing  (e.g. 'shear/40_1pct_0.3ptb')
# Both tags are consumed in Cell 4 when building the entries list.
# ------------------------------------------------------------------
total_data = []
print(f"{'File':<40} {'Structures':>12}")
print("-" * 54)
for label, path in files.items():
    data     = loadfn(path)
    eref_key = get_eref_key(label)
    strain_pct, perturbation, direction = get_strain_metadata(label)
    for d in data:
        d["_eref_key"]    = eref_key      # composition key e.g. 'BiSbTe40'
        d["_file_label"]  = label         # full label e.g. 'shear/20_3pct_0.3ptb'
        d["_strain_pct"]  = strain_pct    # e.g. 3.0 or None
        d["_perturbation"]= perturbation  # e.g. 0.3 or 0.0
        d["_direction"]   = direction     # e.g. 'X' or None
    print(f"{label:<40} {len(data):>12}")
    total_data += data
    
print("-" * 54)
print(f"{'TOTAL':<40} {len(total_data):>12}")

# ------------------------------------------------------------------
# Extract arrays and subtract the composition-matched reference energy
# from each structure's DFT total energy.
#
# total_energies = E_DFT - sum(n_X * E_atom[X])   (eV)
#
# Forces and stresses are NOT modified — they are already physical.
# ------------------------------------------------------------------
total_structures   = [d["structure"]                for d in total_data]
total_forces       = [d["outputs"]["forces"]        for d in total_data]
total_stresses     = [d["outputs"]["virial_stress"] for d in total_data]  # GPa, Voigt-6

total_energies_raw = np.array([d["outputs"]["energy"]  for d in total_data])  # raw DFT (eV)
total_eref         = np.array([eref_90[d["_eref_key"]] for d in total_data])  # reference (eV)
total_energies     = total_energies_raw - total_eref   # cohesive energies for NEP training

# Sanity check — should be small negative values (~-3 to -4 eV/atom × 90 atoms)
print(f"\nEnergy after reference subtraction:")
print(f"  min  : {total_energies.min():.4f} eV")
print(f"  max  : {total_energies.max():.4f} eV")
print(f"  mean : {total_energies.mean():.4f} eV")
print(f"  mean/atom: {total_energies.mean()/90:.4f} eV/atom")

File                                       Structures
------------------------------------------------------
random_disp/Bi2Te3                                200
random_disp/BiTeSb_20                             200
random_disp/BiTeSb_40                             200
random_disp/BiTeSb_60                             200
random_disp/BiTeSb_80                             200
random_disp/Sb2Te3                                200
shear/Bi2Te3_1pct_0.3ptb                           50
shear/Bi2Te3_1pct_0ptb                             50
shear/Bi2Te3_3pct_0.3ptb                           50
shear/Bi2Te3_3pct_0ptb                             50
shear/20_1pct_0.3ptb                               50
shear/20_1pct_0ptb                                 50
shear/20_3pct_0.3ptb                               50
shear/20_3pct_0ptb                                 50
shear/40_1pct_0.3ptb                               50
shear/40_1pct_0ptb                                 50
shear/40_3pct_0.3ptb       

In [13]:
# ============================================================
# CELL 4 — Convert stress units and write NEP train/test XYZ
#
# GPUMD/NEP expects the extended-XYZ format with:
#   - energy in eV  (already done in Cell 3)
#   - forces in eV/Å  (already in this unit from QE via pymatgen)
#   - stress as full 3×3 (9 components) in eV/Å³
#
# Input stress is Voigt-6 in GPa (as stored by atomate/pymatgen).
# Steps:
#   1. Flip sign: atomate stores virial stress with opposite sign
#      convention to what NEP expects
#   2. Convert GPa → eV/Å³  (1 GPa = 1/160.21766208 eV/Å³)
#   3. Expand Voigt-6 [xx,yy,zz,yz,xz,xy] → full symmetric 3×3
#
# Metadata written per frame (all OVITO-compatible):
#   config_type   = alloy composition      e.g. BiSbTe40
#   data_type     = dataset category       e.g. shear
#   strain_pct    = applied strain %       e.g. 3.0  (None for random_disp)
#   perturbation  = atomic displacement Å  e.g. 0.3A
#   direction     = strain axis            e.g. X    (uniaxial only)
#   energy_units  = eV
#   forces_units  = eV/Ang
#   stress_units  = eV/Ang3
#
# Train/test split: 90% train / 10% test (shuffled with seed=42)
# ============================================================

import re

# ------------------------------------------------------------------
# 1. Flags and constants
# ------------------------------------------------------------------
FLIP_STRESS_SIGN  = True                    # atomate virial sign → NEP convention
GPA_TO_EV_PER_A3  = 1.0 / 160.21766208     # exact unit conversion

USE_SPLIT   = True   # True → write separate train.xyz and test.xyz
SPLIT_RATIO = 0.9    # fraction of total data used for training

# ------------------------------------------------------------------
# 2. Stress conversion: Voigt-6 GPa → full 9-component eV/Å³
#    Voigt order assumed: [σ_xx, σ_yy, σ_zz, σ_yz, σ_xz, σ_xy]
#    NEP expects row-major flattened 3×3:
#    [xx, xy, xz, yx, yy, yz, zx, zy, zz]
# ------------------------------------------------------------------
def convert_voigt6_gpa_to_ev_per_a3(voigt6):
    factor = -GPA_TO_EV_PER_A3 if FLIP_STRESS_SIGN else GPA_TO_EV_PER_A3
    σxx, σyy, σzz, σyz, σxz, σxy = voigt6
    return [
        σxx*factor, σxy*factor, σxz*factor,
        σxy*factor, σyy*factor, σyz*factor,
        σxz*factor, σyz*factor, σzz*factor,
    ]

# ------------------------------------------------------------------
# 3. Metadata parser — extracts strain %, perturbation, direction
#    from the _file_label string set in Cell 3.
#
#    Examples:
#      'shear/20_3pct_0.3ptb'   → strain_pct=3.0,  perturbation=0.3, direction=None
#      'shear/Bi2Te3_1pct_0ptb' → strain_pct=1.0,  perturbation=0.0, direction=None
#      'uniaxial/60_X'          → strain_pct=None, perturbation=0.0, direction='X'
#      'random_disp/BiTeSb_40'  → strain_pct=None, perturbation=0.3, direction=None
# ------------------------------------------------------------------
def get_strain_metadata(file_label):
    tag   = file_label.split("/")[-1]   # e.g. '20_3pct_0.3ptb', 'Bi2Te3_X'
    dtype = file_label.split("/")[0]    # 'random_disp', 'shear', 'uniaxial'

    strain_pct   = None
    perturbation = 0.0
    direction    = None

    if dtype == "random_disp":
        # fixed 0.3 Å random displacement, no strain applied
        perturbation = 0.3

    elif dtype == "shear":
        # parse strain magnitude e.g. "3pct" → 3.0
        m_strain = re.search(r'(\d+)pct', tag)
        if m_strain:
            strain_pct = float(m_strain.group(1))
        # parse perturbation amplitude e.g. "0.3ptb" → 0.3, "0ptb" → 0.0
        m_ptb = re.search(r'_(\d+\.?\d*)ptb', tag)
        if m_ptb:
            perturbation = float(m_ptb.group(1))

    elif dtype == "uniaxial":
        # parse direction e.g. "60_X" → 'X'
        m_dir = re.search(r'_(X|Y|Z)$', tag)
        if m_dir:
            direction = m_dir.group(1)
        perturbation = 0.0   # uniaxial has no atomic perturbation

    return strain_pct, perturbation, direction

# ------------------------------------------------------------------
# 4. Sanity check: all arrays must have the same length
# ------------------------------------------------------------------
N = len(total_structures)
assert len(total_energies) == N, "energy count mismatch"
assert len(total_forces)   == N, "forces count mismatch"
assert len(total_stresses) == N, "stress count mismatch"

# ------------------------------------------------------------------
# 5. Pack into entries — each entry carries full metadata + outputs
#
#   config_type  : alloy composition    from _eref_key   e.g. "BiSbTe40"
#   data_type    : dataset category     from _file_label e.g. "shear"
#   strain_pct   : % strain applied     parsed from label e.g. 3.0 / None
#   perturbation : atomic displ. (Å)   parsed from label e.g. 0.3 / 0.0
#   direction    : uniaxial axis        parsed from label e.g. "X" / None
# ------------------------------------------------------------------
entries = []
for i, d in enumerate(total_data):
    config_type                      = d["_eref_key"]
    data_type                        = d["_file_label"].split("/")[0]
    strain_pct, perturbation, direction = get_strain_metadata(d["_file_label"])
    entries.append({
        "structure"   : total_structures[i],
        "config_type" : config_type,
        "data_type"   : data_type,
        "strain_pct"  : strain_pct,
        "perturbation": perturbation,
        "direction"   : direction,
        "outputs": {
            "energy": total_energies[i],                                   # eV (ref subtracted)
            "forces": total_forces[i],                                     # eV/Å
            "stress": convert_voigt6_gpa_to_ev_per_a3(total_stresses[i])  # eV/Å³, 9-component
        }
    })

# ------------------------------------------------------------------
# 6. XYZ writer — one structure per frame, extended-XYZ format
#
#    Header line contains all metadata fields + units explicitly:
#      lattice        : Å, row-major 3×3
#      energy         : eV (isolated-atom reference subtracted)
#      stress         : eV/Å³, full 3×3 symmetric
#      config_type    : alloy composition
#      data_type      : dataset category
#      strain_pct     : % strain (None if not applicable)
#      perturbation   : atomic displacement amplitude in Å
#      direction      : strain axis for uniaxial (None otherwise)
#      energy_units   : eV
#      forces_units   : eV/Ang
#      stress_units   : eV/Ang3
#
#    Atom lines:
#      symbol  x  y  z   fx  fy  fz
# ------------------------------------------------------------------
def write_entry_xyz(f, struct, out, config_type, data_type,
                    strain_pct, perturbation, direction):
    n = len(struct)
    f.write(f"{n}\n")

    lat_str = " ".join(f"{v:.9f}" for v in struct.lattice.matrix.flatten())
    e_str   = f"{out['energy']:.12f}"
    s_str   = " ".join(f"{x:.12f}" for x in out['stress'])

    # format optional fields — always written so every frame is self-describing
    strain_str = f"strain_pct={strain_pct}"    if strain_pct  is not None else "strain_pct=None"
    ptb_str    = f"perturbation={perturbation}A"
    dir_str    = f"direction={direction}"       if direction   is not None else "direction=None"

    header = [
        f'lattice="{lat_str}"',               # Å
        f'energy={e_str}',                    # eV (ref subtracted)
        f'stress="{s_str}"',                  # eV/Å³
        f'config_type={config_type}',         # e.g. BiSbTe40
        f'data_type={data_type}',             # e.g. shear
        strain_str,                           # e.g. strain_pct=3.0
        ptb_str,                              # e.g. perturbation=0.3A
        dir_str,                              # e.g. direction=X
        'energy_units=eV',                    # explicit unit tag
        'forces_units=eV/Ang',                # explicit unit tag
        'stress_units=eV/Ang3',               # explicit unit tag
        'properties=species:S:1:pos:R:3:forces:R:3'
    ]
    f.write(" ".join(header) + "\n")

    coords = struct.cart_coords
    for idx, (fx, fy, fz) in enumerate(out['forces']):
        sym     = struct.sites[idx].species.elements[0].symbol
        x, y, z = coords[idx]
        f.write(f"{sym:<2s} {x: .6f} {y: .6f} {z: .6f}   {fx: .6f} {fy: .6f} {fz: .6f}\n")

# ------------------------------------------------------------------
# 7. Shuffle with fixed seed for reproducibility, then split and write
# ------------------------------------------------------------------
random.seed(42)         # fixed seed → same train/test split every run
random.shuffle(entries)

out_dir = "/home/ashwani/BiSbTe_AlloyTransport/training_data_nep_format"

if USE_SPLIT:
    split_idx     = int(SPLIT_RATIO * N)
    train_entries = entries[:split_idx]
    test_entries  = entries[split_idx:]

    def write_dataset(fname, data):
        with open(fname, 'w') as f:
            for e in data:
                write_entry_xyz(
                    f,
                    e['structure'],
                    e['outputs'],
                    e['config_type'],
                    e['data_type'],
                    e['strain_pct'],
                    e['perturbation'],
                    e['direction']
                )
        print(f"Wrote {len(data)} entries to '{fname}'")

    write_dataset(f"{out_dir}/train.xyz", train_entries)
    write_dataset(f"{out_dir}/test.xyz",  test_entries)

else:
    # Write all data as a single file (no split)
    out_file = f"{out_dir}/all_ref_E.xyz"
    with open(out_file, 'w') as f:
        for e in entries:
            write_entry_xyz(
                f,
                e['structure'],
                e['outputs'],
                e['config_type'],
                e['data_type'],
                e['strain_pct'],
                e['perturbation'],
                e['direction']
            )
    print(f"Wrote {len(entries)} entries to '{out_file}'")


Wrote 2293 entries to '/home/ashwani/BiSbTe_AlloyTransport/training_data_nep_format/train.xyz'
Wrote 255 entries to '/home/ashwani/BiSbTe_AlloyTransport/training_data_nep_format/test.xyz'
